# KingFisher Flex

```{device-card} thermo-fisher-kingfisher-flex
```

Run computer-hosted BindIt BDZ exports with a matching **96DW head, tip comb and plates**.
PLR validates the export, binds its plates to resource objects, and streams its generated
commands over USB HID. BindIt does not need to be running during execution.

**Experimental:** hardware checks cover firmware 1.01.0 on macOS, including mixing,
bead collection/release, drying, 37 C heating, timed pauses, loops and host confirmations.
Resource-aware ManualOperator host and physical-Start handling have passed two-plate
loading, mixing and unloading runs, including inventory checks after all four transfers.
The physical-Start check used a diagnostic with mixing shortened to 20 seconds.
A carousel-only reagent-addition diagnostic also passed: the host handler identified
the bound plate and presented holder, awaited confirmation of a 50 uL-per-well water
addition, and verified unchanged plate occupancy before continuing. Robotic transfers
remain application-owned. Command completion does not verify tip-comb
attachment; inspect pickup and clearances. Other head/plate formats, onboard
protocol management and generation of protocols without a BDZ are outside this driver.

See [installation](../installation.md) and [command-line tools](../cli.md).


## Physical setup

Prepare the 96DW head, matching tips and supported plates according to the instrument
instructions. Close BindIt before connecting. Run hardware cells individually.


## Inspect a protocol offline

Set the path to your BindIt export. Reading and preflight do not open USB.
`compatibility_issues()` lists blockers; `preflight()` raises if any remain.
`prompts()` shows plate actions and reagent additions. Plate labels are not tray numbers.


Use an executable export produced by BindIt for the installed 96DW configuration.
The driver retains the exported motions, timing and heating commands; it does not
calculate coordinates from volumes or reconstruct a method from step names. An
unsupported export is rejected as a whole. Passing validation does not prove the
physical setup, plate contents or protocol outcome.


In [ ]:
from pathlib import Path
from pylabrobot.thermo_fisher.kingfisher.flex import KingFisherFlex, read_flex_bdz

protocol = read_flex_bdz(Path("protocol.bdz"))
commands = protocol.preflight()
print(protocol.name, protocol.sha256, len(commands))
print([(plate.name, plate.id) for plate in protocol.plates])
protocol.prompts()

## Logical carousel resources

`flex.carousel` is a `PlateCarrier` with eight persistent `PlateHolder` sites,
keyed **1–8** to match the printed trays. Each tray holds at most one PLR `Plate`.
Remove its current plate explicitly before assigning another. Assignment and
unassignment only update the resource tree; they do not move hardware.

The following offline example assigns a KingFisher plate, then removes it.
Plate compatibility with a BDZ is a separate execution check.

Carousel dimensions and pedestal heights are **unmeasured zero placeholders**;
holder locations are undefined. Local plate coordinates are organizational only.
This model is not calibrated for robot transfers or collision checking.

In [ ]:
from pylabrobot.resources import thermo_TS_kingfisher_96_wellplate_2000uL_Vb

flex = KingFisherFlex(name="flex_a")
plate = thermo_TS_kingfisher_96_wellplate_2000uL_Vb("sample_plate")
tray = flex.carousel[2]
tray.assign_child_resource(plate)  # Equivalent to flex.carousel[2] = plate.
assert plate.parent is tray
assert tray.resource is plate
plate.unassign()  # Equivalent to flex.carousel[2] = None.

`processing_slot` and `loading_slot` identify the trays at the two stations;
`processing_holder` and `loading_holder` return those same holder objects.
They are `None` before initialization, during rotation, and after interruption or
disconnect. Orientation updates only after successful motion completion. Rotation
does not move plates between holders.

Plate assignments persist through initialization, disconnect and failures.
`inventory_verified` starts false: an empty resource tree does not prove the
physical carousel is empty. Reconcile inventory after connection and before
execution, as shown below. Homing only establishes orientation.

BDZ execution requires explicit plate identity bindings. Binding does not assign
plates to holders. Action handlers perform transfers and update the resource tree;
the driver checks the result before continuing. Resource changes outside an active
action invalidate inventory certainty. Reconnection, interruption and disconnection
also require a fresh inventory reconciliation.

## Bind every protocol plate before execution

Choose the PLR `Plate` object representing each physical plate required by this
BDZ. Keys must exactly match the names in `protocol.plates`, including case;
PLR resource names can differ. Include the empty tip-comb pickup/return plate.

The example below assumes a four-plate export with these names. Replace the
mapping with all names from your selected BDZ and your corresponding PLR plate
objects. Use existing objects when those plates are already modeled elsewhere
in your workcell; do not create another object for the same physical plate.

`bind_plates()` runs offline and rejects missing or unknown names, duplicate BDZ
names, non-plate resources and reuse of one PLR object for different BDZ plates.
For duplicate BDZ names, give the plates distinct names in BindIt and re-export.
The resulting run retains its selected export and a read-only mapping in
`bound_protocol.plates_by_id`. Editing the input dictionary cannot change it.
Preparing another protocol requires its own bindings.

Binding records identity only: it does not rename resources, move plates, validate
physical plate geometry or prove which plate an operator has loaded.

In [ ]:
tip_plate = thermo_TS_kingfisher_96_wellplate_2000uL_Vb("tips_001")
bead_plate = thermo_TS_kingfisher_96_wellplate_2000uL_Vb("beads_001")
wash_plate = thermo_TS_kingfisher_96_wellplate_2000uL_Vb("wash_001")
resuspension_plate = thermo_TS_kingfisher_96_wellplate_2000uL_Vb("resuspension_001")

bound_protocol = protocol.bind_plates({
  "Empty Tip Comb Plate": tip_plate,
  "Bead Plate": bead_plate,
  "Wash Plate": wash_plate,
  "Resuspension Plate": resuspension_plate,
})

## Connect and identify

`setup()` opens USB and reads the identity without moving the instrument.
Pass `serial_number="..."` to select among multiple instruments.

The `name` argument distinguishes resource names; `serial_number` selects the USB instrument.
For two instruments, use distinct names and their respective serial numbers.


In [ ]:
await flex.setup()
(flex.version, flex.serial)

## Initialize

**This moves the instrument**, disables heating, and presents tray 4 at loading
with tray 1 at processing.


In [ ]:
await flex.initialize()
(flex.processing_slot, flex.loading_slot)

## Reconcile initial inventory

Inspect all eight physical trays and make the resource assignments match them.
For a preloaded plate, use `flex.carousel[tray_number] = bound_plate`; remove stale
assignments with `plate.unassign()`. The bound object must match the expected tray
in the BDZ loading sequence. Empty trays must be checked too.

Only after that inspection, execute the confirmation below. It sends no instrument
commands. Running before reconciliation raises an error without starting the method.
Plates can be loaded progressively through action handlers; they need not all be
present before the run. A successful checked run preserves verified inventory.
After interruption, inspect retained assignments, establish a fresh session and
reconcile again before attempting another run.

In [ ]:
flex.reconcile_inventory()
flex.inventory_verified

## Observe events

Start one event consumer before running. Events include temperature readings,
plate requests, pauses, confirmations, failures, and run completion. Temperature
readings describe heater and ambient sensors, not well-liquid temperature.


In [ ]:
import asyncio
import logging

logging.basicConfig(level=logging.INFO)


async def observe_events():
  async for event in flex.events():
    logging.info("%s: %s; prompt=%s", event.kind, event.text, event.prompt)


observer = asyncio.create_task(observe_events())

## Software/manual action handling

A `FlexAction` supplies the bound `plate`, presented `holder`, BDZ `plate_id` and
`plate_name`, instructions in `label`, reagent `additions`, and the source command,
line number and export fingerprint. A generic pause can have no bound plate.

`FlexManualHandler` uses [ManualOperator](../../../machine-agnostic-features/manual-operator.md):
`move_resource()` handles transfers, and `perform()` handles interventions and
confirmations of preloaded plates. Loading uses the plate's current parent. Your
application supplies the unloading destination. The operator waits for its provider
before updating resource assignments, and the Flex checks occupancy before advancing.

Here the console provider acknowledges completion when you press Enter. You can
substitute any `OperatorActionProvider`, such as one backed by your application's UI.
Finish the action and clear the carousel before acknowledging. Rejection, failure,
timeout and cancellation prevent further execution and require recovery.

The logical staging holders below are an example of modeled sources and unloading
destinations. Replace them with your actual workcell resources. Their zero dimensions
are unmeasured placeholders, not robot coordinates. Existing parent assignments are
preserved. A manual load requires a modeled source; binding alone does not supply one.

In [ ]:
from pylabrobot.manual_operator import ConsoleOperatorActionProvider, ManualOperator
from pylabrobot.resources import Resource, ResourceHolder
from pylabrobot.thermo_fisher.kingfisher.flex import FlexAction, FlexManualHandler

staging = {
  plate_id: ResourceHolder(
    name=f"staging_{plate.name}", size_x=0, size_y=0, size_z=0,
    metadata={"geometry": "unmeasured"},
  )
  for plate_id, plate in bound_protocol.plates_by_id.items()
}
for plate_id, plate in bound_protocol.plates_by_id.items():
  if plate.parent is None:
    staging[plate_id].assign_child_resource(plate)


def unloading_destination(action: FlexAction) -> Resource:
  assert action.plate_id is not None
  return staging[action.plate_id]


operator = ManualOperator(ConsoleOperatorActionProvider(), name="flex_operator")
manual_handler = FlexManualHandler(operator, unload_destination=unloading_destination)

## Run a local protocol with software acknowledgment

This starts motion and may start heating. Complete each requested transfer or
intervention before confirming through the provider. The handler updates the model;
plate assignment events alone never advance execution. Reagent requests include
reagent names and volumes per well.

Host acknowledgment omits PAU, so no physical Start press is needed. An unbound
protocol or unreconciled inventory is rejected before execution commands are sent.
`operator_timeout=None` permits indefinite operator waits while session failures
remain monitored. Temperature setpoints do not themselves wait for target attainment.
The call returns after execution and finalization.

Only `handle_action` fulfills requests. Use `action.action` to dispatch loading,
unloading and interventions to different application operations. `flex.events()` is
an observation stream; consuming an event does not acknowledge or complete an action.
After the handler returns exactly `True`, the driver checks all eight holders against
the expected post-action inventory. It does not independently sense plate transfers.


In [ ]:
result = await flex.run_protocol(
  bound_protocol,
  handle_action=manual_handler,
  acknowledgement="host",
  operator_timeout=None,
)
result

## Alternative: physical Start through ManualOperator

Use this path instead of the software-acknowledged run above when you want the
instrument's Start button to acknowledge the actions. Each request gets its own
`FlexStartAcknowledgement` provider, bound to that exact active action. It sends
the original PAU and waits for `PAU END 0` within the ManualOperator lifecycle.
Transfers are recorded after that completion, and the Flex still checks occupancy
before issuing the next command. The provider logs transfer instructions and
ManualOperator publishes lifecycle events; no console Enter press is also required.

`acknowledgement="instrument"` must be selected before PAU. The adapter cannot
release an already active pause or reuse an acknowledgment for another action.
Firmware completion is not independent verification of physical plate identity.

In [ ]:
from pylabrobot.thermo_fisher.kingfisher.flex import FlexStartAcknowledgement


async def physical_handler(action: FlexAction) -> bool:
  operator = ManualOperator(
    FlexStartAcknowledgement(flex, action), name="flex_physical_operator"
  )
  handler = FlexManualHandler(operator, unload_destination=unloading_destination)
  return await handler(action)


result = await flex.run_protocol(
  bound_protocol,
  handle_action=physical_handler,
  acknowledgement="instrument",
  operator_timeout=None,
)

## Application-selected handling

`acknowledgement` can also be a synchronous function returning `"host"` or
`"instrument"` for each `FlexAction`. Supply a handler that uses the corresponding
provider. For example, transfer actions can use software confirmation while protocol
interventions use physical Start. Selection happens before that action's PAU is sent.

A robotic handler uses `"host"` acknowledgment and awaits its own transfer or
intervention implementation. That implementation must update the resource tree and
finish clearing the carousel before the handler returns `True`. The Flex driver
selects no arm, storage location, unloading destination or calibrated robot path.
Neither ManualOperator nor any specific arm is mandatory for execution.

## Hardware-free robotic-handler example

This example exercises only the action interface and resource model. It opens no USB
connection and operates no robot. `simulated_transfer` stands in for an application
transfer operation that returns after motion and model updates finish. Replace it
with a calibrated implementation for a real workcell; these holders have no calibrated
geometry. The synthetic requests are not an executable instrument protocol.

In [ ]:
from dataclasses import replace
from pylabrobot.thermo_fisher.kingfisher.flex import FlexCommand

example_flex = KingFisherFlex(name="handler_example")
example_plate = thermo_TS_kingfisher_96_wellplate_2000uL_Vb("example_plate")
example_source = ResourceHolder("example_source", 0, 0, 0)
example_destination = ResourceHolder("example_destination", 0, 0, 0)
example_source.assign_child_resource(example_plate)


async def simulated_transfer(resource: Resource, destination: Resource) -> None:
  """Simulate an awaited transfer with completed model updates; no hardware."""
  await asyncio.sleep(0)
  assert resource.parent is not None
  resource.unassign()
  destination.assign_child_resource(resource, location=None)


async def robotic_handler(action: FlexAction) -> bool:
  """Application-owned fulfillment; substitute real awaited operations as appropriate."""
  if action.action == "pause":
    await asyncio.sleep(0)  # Simulated intervention completion.
  else:
    assert action.plate is not None
    destination = action.holder if action.action == "load" else example_destination
    await simulated_transfer(action.plate, destination)
  return True


example_action = FlexAction(
  action="load", label="EXAMPLE", plate_id="example-id", plate_name="EXAMPLE",
  plate=example_plate, holder=example_flex.carousel[2],
  processing_slot=7, loading_slot=2, line_number=1,
  command=FlexCommand(1, 'PAU 1 "EXAMPLE"'), source_sha256="0" * 64,
)
await robotic_handler(example_action)
assert example_plate.parent is example_flex.carousel[2]
await robotic_handler(replace(
  example_action, action="unload", line_number=2,
  command=FlexCommand(2, 'PAU 2 "EXAMPLE"'),
))
assert example_plate.parent is example_destination

## Abort and recovery

Call `await flex.abort()` from another task while execution is pending. Keep the run
in an `asyncio.Task` when it needs a supervising task. Aborting can move axes.
Canceling a Python task, interrupting a notebook cell, or calling `stop()` does not
stop the instrument or guarantee heater shutdown.

After a rejected action, timeout, fault, abort or disconnect, further execution is
blocked. The driver preserves the last modeled plate assignments but invalidates
orientation and inventory verification. A failed transfer may have physically completed
without its model update, so retained assignments are evidence to inspect, not a
physical inventory reading.

Before restarting:

1. Ensure motion has stopped and the instrument can be safely inspected. Check the
   tip comb, plates and heater; use the instrument's recovery procedure as needed.
2. Call `await flex.stop()` to close the failed session, then `await flex.setup()`
   to establish a fresh connection.
3. Correct the resource tree to match the observed plates, including external sources
   and destinations. Initialize only when the instrument is safe to home, using
   `await flex.initialize()`; homing does not empty trays or fix an attached tip comb.
4. Inspect all eight trays and call `flex.reconcile_inventory()`. Recheck your plate
   bindings and handler before explicitly starting another run.

There is no automatic retry or mid-protocol resume. Restarting executes the method
from its beginning and can repeat chemical operations.


## Disconnect

`stop()` closes USB and ends the event stream. It does not stop motion.


In [ ]:
await flex.stop()
await observer